In [0]:
__source_path = "/Volumes/fraudwatch/source/watchlist/source_data/"
__schema_path = "/Volumes/fraudwatch/source/watchlist/schema/"
__checkpoint_path = "/Volumes/fraudwatch/source/watchlist/checkpoints/"

In [0]:
dbutils.fs.ls(__source_path)

In [0]:
input_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", __schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(__source_path)
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

transformed_df = input_stream.select(
    "*",
    col("_metadata.file_path").alias("file_path"),
    current_timestamp().alias("ingestion_ts")
)

In [0]:
query = (
    transformed_df.writeStream.format("delta")
    .outputMode("Append")
    .option("checkpointLocation", __checkpoint_path)
    .trigger(availableNow=True)
    .toTable("fraudwatch.bronze.watchlist_batch_test")
)

In [0]:
%sql
select * from fraudwatch.bronze.watchlist_batch_test;

In [0]:
df = spark.read.table("fraudwatch.bronze.watchlist_batch_test")
df.columns